In [1]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [9]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [4]:
# Import datasets
datasets = {}
dataset_names = ['clfever', 'phemeplus', 'vitc']
for dataset_name in dataset_names:
    with open(f'{dataset_name}.json') as f:
        datasets[dataset_name] = json.load(f)

In [5]:
# Populate Vercel KV with datasets
for dataset_name in dataset_names:
    dataset = datasets[dataset_name]
    for datapoint in dataset:
        id = datapoint['claim_id']
        # r.hset(id, mapping={
        #     'claim': datapoint['claim'],
        #     'evidence': datapoint['evidence'],
        #     'label': datapoint['label']
        # })

In [54]:
# Create batches containing 25 datapoints each
batch_ids = []
for dataset_name in dataset_names:
    dataset = datasets[dataset_name]
    for i in range(0, len(dataset), 25):
        batch_dataset = dataset[i:i+25]
        claim_ids = []
        for batch_datapoint in batch_dataset:
            id = batch_datapoint['claim_id']
            claim_ids.append(id)
        batch_id = f'batch_{dataset_name}_{i//25 + 1}'
        batch_ids.append(batch_id)
        r.hset(batch_id, mapping={'claim_ids': json.dumps(claim_ids)})   

In [13]:
# Import assessesment samples
vitc_assessment = pd.read_csv('Assesment samples - VitC.csv')

In [28]:
vitc_assessment

,Id,claim,evidence,Ground Truth,Label,Unnamed: 5
0,asses_1_vitc,There have been more than five confirmed cases...,"On 25 January 2020 , the number of laboratory-...",SUPPORTS,d,NaN
1,asses_2_vitc,The most prominent smartphone vendor in the wo...,BlackBerry was one of the most prominent smart...,SUPPORTS,a,This one is in the guideline
2,asses_3_vitc,"According to expectations , the Boeing B-52 St...","After being upgraded between 2013 and 2015 , i...",REFUTES,d,NaN
3,asses_4_vitc,Marcus Bentley is a British chef .,"Marcus Morgan Bentley ( born October 4 , 1967 ...",SUPPORTS,d,NaN
4,asses_5_vitc,"Before March 29 , 2020 , Nevada had less than ...","As of March 29 , 2020 , 738 positive cases and...",REFUTES,a,NaN


In [30]:
# iterate through rows of dataframe
assess_ids = []
for index, row in vitc_assessment.iterrows():
    id = row["Id"]
    assess_ids.append(id)
    claim = row["claim"]
    evidence = row["evidence"]
    label = row["Ground Truth"]
    reasoning = row["Label"]
    if reasoning == 'a':
        reasoning = 'abductive'
    elif reasoning == 'd':
        reasoning = 'deductive'
    
    r.hset(id, mapping={
        'claim': claim,
        'evidence': evidence,
        'label': label,
        'reasoning': reasoning
    })


In [33]:
# create batch for assessement
r.hset('assess_vitc', mapping={'claim_ids': json.dumps(assess_ids)}) 

1

In [34]:
r.hgetall('assess_vitc')

{b'claim_ids': b'["asses_1_vitc", "asses_2_vitc", "asses_3_vitc", "asses_4_vitc", "asses_5_vitc"]'}

In [35]:
# Delete queue 
r.delete('queue')

0

In [37]:
vitc_only_batch = [batch for batch in batch_ids if 'vitc' in batch]

In [42]:
# Create queue 
r.lpush('queue', *vitc_only_batch)

20

In [45]:
queue = r.lrange('queue', 0, -1)
print(len(queue))
print(queue)

18
[b'batch_vitc_18', b'batch_vitc_17', b'batch_vitc_16', b'batch_vitc_15', b'batch_vitc_14', b'batch_vitc_13', b'batch_vitc_12', b'batch_vitc_11', b'batch_vitc_10', b'batch_vitc_9', b'batch_vitc_8', b'batch_vitc_7', b'batch_vitc_6', b'batch_vitc_5', b'batch_vitc_4', b'batch_vitc_3', b'batch_vitc_2', b'batch_vitc_1']


In [56]:
r.hgetall('batch_vitc_20')

{b'claim_ids': b'["vitc_16915", "vitc_15819", "vitc_8186", "vitc_12391", "vitc_3532", "vitc_16455", "vitc_5317", "vitc_13761", "vitc_17114", "vitc_9659", "vitc_16073", "vitc_14246", "vitc_17686", "vitc_8802", "vitc_12141", "vitc_18578", "vitc_4565", "vitc_18936", "vitc_8683", "vitc_5167", "vitc_17640", "vitc_16311", "vitc_1159", "vitc_4783", "vitc_17988"]'}

In [60]:
# delete participants
# r.delete('participants')

1

In [62]:
# Get list og participants who completed the task
r.lrange('participants',0,-1)

[b'{"participant":"test_success","batchId":"batch_vitc_6","stage":"annotation"}',
 b'{"participant":"test_success","batchId":"batch_vitc_6","stage":"assessment"}']

In [63]:
# get answers from specific participant
r.hgetall('test_success')

{b'vitc_8199': b'deductive',
 b'vitc_10093': b'deductive',
 b'asses_4_vitc': b'deductive',
 b'vitc_15763': b'deductive',
 b'vitc_426': b'abductive',
 b'vitc_254': b'deductive',
 b'vitc_6241': b'deductive',
 b'vitc_19345': b'abductive',
 b'vitc_17871': b'deductive',
 b'vitc_14127': b'abductive',
 b'asses_2_vitc': b'abductive',
 b'vitc_11728': b'abductive',
 b'vitc_14727': b'abductive',
 b'vitc_13108': b'deductive',
 b'vitc_2496': b'abductive',
 b'vitc_18759': b'abductive',
 b'vitc_2393': b'deductive',
 b'vitc_3601': b'abductive',
 b'vitc_15420': b'deductive',
 b'vitc_14882': b'abductive',
 b'vitc_4206': b'deductive',
 b'vitc_11951': b'deductive',
 b'vitc_16995': b'abductive',
 b'vitc_10194': b'abductive',
 b'vitc_2811': b'deductive',
 b'vitc_1550': b'deductive',
 b'vitc_619': b'deductive',
 b'vitc_10381': b'abductive',
 b'vitc_1842': b'deductive',
 b'vitc_6166': b'deductive',
 b'asses_3_vitc': b'deductive',
 b'vitc_13708': b'deductive',
 b'vitc_8090': b'deductive',
 b'vitc_4509': b'abdu